# 쇼핑몰 상품 추천 RAG v0 Starter

이 노트북은 **상품데이터와 거래데이터가 1:1로 매칭되지 않는 상황**을 전제로 만든 상품 추천 v0입니다.

## 핵심 방향

```text
상품데이터 = 현재 추천 가능한 상품 후보 DB
거래데이터 = 과거 구매 패턴/업종/행사 힌트 DB
```

따라서 두 파일을 억지로 상품명 매칭하지 않습니다.

## v0 추천 구조

```text
사용자 질문
↓
예산/수량 조건 추출
↓
거래데이터에서 비슷한 과거 사례 검색
↓
과거 사례의 상품분류 힌트 추출
↓
상품데이터에서 가격/수량 조건 반영
↓
상품명/카테고리/키워드 기반 검색
↓
Top 5 추천
```

> 이 v0는 완성형 추천기가 아니라, 상품 추천 RAG 구조를 검증하기 위한 실험용 노트북입니다.


## 0. 패키지 설치

처음 한 번만 실행하면 됩니다.


In [ ]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn


## 1. 기본 설정 및 파일 경로

샘플 파일 기준으로 설정되어 있습니다.

원본 대용량 파일로 바꾸려면 `PRODUCT_PATH`, `TRADE_PATH`만 수정하면 됩니다.


In [ ]:
from pathlib import Path
import re
import html
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

# =========================
# 파일 경로
# =========================
PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

# ChatGPT 샌드박스 경로에서 실행하는 경우
if not PRODUCT_PATH.exists():
    PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
if not TRADE_PATH.exists():
    TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 2. 엑셀 로딩 및 컬럼 확인

현재 샘플 기준:

- 거래데이터: 100행 × 23컬럼
- 상품데이터: 100행 × 93컬럼

컬럼명이 다르면 아래 컬럼 매핑 부분에서 수정하면 됩니다.


In [ ]:
# =========================
# 엑셀 로딩
# =========================

product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)

print("\n[상품데이터 컬럼]")
for i, col in enumerate(product_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n[거래데이터 컬럼]")
for i, col in enumerate(trade_raw.columns, start=1):
    print(f"{i:02d}. {col}")

display(product_raw.head(3))
display(trade_raw.head(3))


## 3. 대용량 원본 샘플링 코드

나중에 원본 파일이 너무 클 때 사용하는 샘플링 코드입니다.

단순 랜덤 샘플만 뽑으면 중요한 카테고리가 빠질 수 있으므로,

```text
랜덤 샘플
+
카테고리별 균등 샘플
```

을 섞는 방식입니다.


In [ ]:
# =========================
# 대용량 원본 샘플링 함수
# =========================

def make_sample_excel(
    input_path,
    output_path,
    n_random=300,
    group_cols=None,
    n_per_group=5,
    random_state=42,
):
    """
    대용량 엑셀에서 샘플 파일을 생성합니다.

    input_path: 원본 엑셀 경로
    output_path: 저장할 샘플 엑셀 경로
    n_random: 전체 랜덤 샘플 수
    group_cols: 카테고리 균등 샘플 기준 컬럼 리스트
    n_per_group: 그룹별 샘플 수
    """

    df = pd.read_excel(input_path, dtype=str).fillna("")
    print("원본 행 수:", len(df))
    print("원본 컬럼 수:", len(df.columns))

    samples = []

    # 1) 전체 랜덤 샘플
    if len(df) > 0:
        random_sample = df.sample(
            n=min(n_random, len(df)),
            random_state=random_state,
        )
        samples.append(random_sample)

    # 2) 그룹별 균등 샘플
    if group_cols:
        valid_group_cols = [c for c in group_cols if c in df.columns]
        if valid_group_cols:
            grouped_sample = (
                df.groupby(valid_group_cols, dropna=False, group_keys=False)
                .apply(lambda x: x.sample(n=min(n_per_group, len(x)), random_state=random_state))
            )
            samples.append(grouped_sample)
        else:
            print("주의: group_cols에 지정한 컬럼이 파일에 없습니다:", group_cols)

    if samples:
        sample_df = pd.concat(samples, ignore_index=True).drop_duplicates()
    else:
        sample_df = df.head(n_random).copy()

    sample_df.to_excel(output_path, index=False)
    print("샘플 저장 완료:", output_path)
    print("샘플 행 수:", len(sample_df))

    return sample_df


# 사용 예시 1: 상품데이터 샘플링
# sample_product = make_sample_excel(
#     input_path="상품데이터_원본.xlsx",
#     output_path="상품데이터_샘플.xlsx",
#     n_random=300,
#     group_cols=["대 카테고리", "중 카테고리"],
#     n_per_group=5,
# )

# 사용 예시 2: 거래데이터 샘플링
# sample_trade = make_sample_excel(
#     input_path="거래데이터_원본.xlsx",
#     output_path="거래데이터_샘플.xlsx",
#     n_random=300,
#     group_cols=["구매처 분류(대)", "구매처 분류(중)", "상품분류(중)"],
#     n_per_group=5,
# )


## 4. 전처리 유틸 함수

상품명, 카테고리, 가격, 최소구매수량 등을 정리합니다.


In [ ]:
# =========================
# 전처리 유틸
# =========================

def clean_text(x):
    """HTML 태그, 과도한 공백 등을 제거합니다."""
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    """가격/수량 문자열을 숫자로 변환합니다."""
    if pd.isna(x):
        return None

    s = str(x).strip()

    if s in ["", "-", "nan", "None", "null"]:
        return None

    s = re.sub(r"[^0-9.]", "", s)

    if s == "":
        return None

    try:
        return float(s)
    except Exception:
        return None


def safe_col(df, col):
    """컬럼이 없을 때도 오류 없이 빈 문자열 Series를 반환합니다."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


## 5. 상품데이터 정규화

샘플 상품데이터에서 추천에 필요한 컬럼만 정리합니다.

주요 사용 컬럼:

- 상품번호
- 상품코드
- 상품명
- 상품판매가
- 최소구매수량
- 검색키워드
- 대 카테고리 / 중 카테고리 / 소 카테고리
- 간략한설명
- 상품내용


In [ ]:
# =========================
# 상품데이터 정규화
# =========================

product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["supply_price"] = safe_col(product_df, "상품공급가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

product_df["search_text"] = (
    product_df["product_name"] + " " +
    product_df["category_path"] + " " +
    product_df["keywords"] + " " +
    product_df["summary"] + " " +
    product_df["detail_text"]
).map(clean_text)

# 상품명이 없는 행 제거
product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
display(product_df[[
    "product_id", "product_name", "price", "moq",
    "category_path", "keywords"
]].head(10))


## 6. 거래데이터 정규화

거래데이터는 직접 추천 상품으로 쓰지 않고, **과거 구매 패턴 힌트**로 사용합니다.

주요 사용 컬럼:

- 구매처 분류(대/중/소/세)
- 구매처 명
- 상품
- 상품분류(대/중/소)
- 대량가격/중간가격/소량가격
- 최소구매수량
- 인쇄 방법


In [ ]:
# =========================
# 거래데이터 정규화
# =========================

trade_df = trade_raw.copy()

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["buyer_type"] = (
    safe_col(trade_df, "구매처 분류(대)") + " > " +
    safe_col(trade_df, "구매처 분류(중)") + " > " +
    safe_col(trade_df, "구매처 분류(소)") + " > " +
    safe_col(trade_df, "구매처 분류(세)")
).map(clean_text)

# 거래데이터에는 같은 이름의 '구매처 명' 컬럼이 중복될 수 있어 실제 있는 컬럼을 우선 사용
buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)

trade_df["trade_search_text"] = (
    trade_df["buyer_type"] + " " +
    trade_df["buyer_name"] + " " +
    trade_df["trade_product_name"] + " " +
    trade_df["trade_category_path"] + " " +
    trade_df["print_method"]
).map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 거래 수:", len(trade_df))
display(trade_df[[
    "buyer_type", "buyer_name", "trade_product_name",
    "trade_category_path", "bulk_price", "middle_price", "small_price",
    "trade_moq", "print_method"
]].head(10))


## 7. 검색 인덱스 만들기

v0에서는 벡터DB 없이 TF-IDF 문자 n-gram 검색을 사용합니다.

한국어는 띄어쓰기나 형태소 분석 문제가 있어서, 초반 baseline으로 문자 n-gram이 안정적입니다.


In [ ]:
# =========================
# 상품/거래 검색 인덱스
# =========================

product_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1,
)

product_matrix = product_vectorizer.fit_transform(product_df["search_text"].fillna("").tolist())

trade_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1,
)

trade_matrix = trade_vectorizer.fit_transform(trade_df["trade_search_text"].fillna("").tolist())

print("상품 matrix:", product_matrix.shape)
print("거래 matrix:", trade_matrix.shape)


## 8. 사용자 질문에서 조건 추출

v0에서는 정규식으로 예산/수량만 간단히 추출합니다.

예:

```text
3천원 이하 → budget = 3000
5만원 → budget = 50000
500개 → quantity = 500
```


In [ ]:
# =========================
# 조건 추출
# =========================

def extract_conditions(query: str) -> dict:
    q = str(query)

    budget = None

    # 예: 3000원, 3,000원
    m = re.search(r"(\d+(?:,\d+)*)\s*원\s*(?:이하|미만|안쪽|대로)?", q)
    if m:
        budget = int(m.group(1).replace(",", ""))

    # 예: 3천원, 3천 원, 3천원대
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원(?:대)?", q)
        if m:
            budget = int(m.group(1)) * 1000

    # 예: 5만원, 5만 원
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None

    # 예: 500개, 1,000개, 500EA
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    return {
        "raw_query": q,
        "budget": budget,
        "quantity": quantity,
    }


# 테스트
for q in [
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    "대학교 행사에서 500개 정도 나눠줄 2,000원대 사은품",
    "회사 창립기념품으로 5만원 이하 고급 상품 추천",
]:
    print(q, "=>", extract_conditions(q))


## 9. 거래데이터에서 과거 구매 힌트 찾기

거래데이터는 상품명 매칭용이 아니라,  
**비슷한 업종/행사/구매처에서 어떤 상품군이 자주 등장했는지** 확인하는 용도로 사용합니다.


In [ ]:
# =========================
# 거래 힌트 검색
# =========================

def get_trade_hints(query: str, top_k: int = 10):
    if len(trade_df) == 0:
        return trade_df.head(0), []

    query_vec = trade_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, trade_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    hints = trade_df.iloc[top_indices].copy()
    hints.insert(0, "trade_score", scores[top_indices])

    # 검색된 거래 사례에서 자주 등장한 상품분류 힌트 추출
    category_candidates = []

    for col in ["trade_category_middle", "trade_category_small"]:
        if col in hints.columns:
            category_candidates.extend([
                clean_text(x)
                for x in hints[col].tolist()
                if clean_text(x)
            ])

    if category_candidates:
        category_hints = (
            pd.Series(category_candidates)
            .value_counts()
            .head(5)
            .index
            .tolist()
        )
    else:
        category_hints = []

    return hints, category_hints


# 테스트
trade_hints, category_hints = get_trade_hints("병원 개원 답례품으로 3천원 이하 상품 추천해줘", top_k=10)

print("거래데이터 기반 카테고리 힌트:", category_hints)
display(trade_hints[[
    "trade_score", "buyer_type", "buyer_name", "trade_product_name",
    "trade_category_path", "print_method"
]].head(10))


## 10. 상품 추천 함수

점수 구조:

```text
최종점수 =
상품 검색 유사도
+ 거래데이터 카테고리 힌트 보너스
+ 예산 조건 보너스
+ 수량 조건 보너스/패널티
```

주의:

- v0에서는 가격 조건을 완전히 엄격하게 적용하지 않습니다.
- 샘플 데이터 100개라서 후보가 너무 적을 수 있기 때문입니다.
- 원본 데이터에서는 가격 필터를 더 강하게 적용할 수 있습니다.


In [ ]:
# =========================
# 상품 추천 함수
# =========================

def recommend_products(
    query: str,
    top_k: int = 5,
    trade_top_k: int = 10,
    min_candidates: int = 10,
):
    conditions = extract_conditions(query)

    candidates = product_df.copy()
    filter_notes = []

    # 1) 예산 조건
    # 샘플 데이터에서는 후보가 너무 적을 수 있어, 후보 수가 충분할 때만 엄격 필터 적용
    if conditions["budget"] is not None and candidates["price"].notna().any():
        budget = conditions["budget"]

        strict_candidates = candidates[
            candidates["price"].notna() &
            (candidates["price"] <= budget)
        ]

        if len(strict_candidates) >= min_candidates:
            candidates = strict_candidates
            filter_notes.append(f"판매가 {budget:,}원 이하 필터 적용")
        else:
            loose_candidates = candidates[
                candidates["price"].notna() &
                (candidates["price"] <= budget * 1.2)
            ]

            if len(loose_candidates) >= min_candidates:
                candidates = loose_candidates
                filter_notes.append(f"후보 부족으로 판매가 {int(budget * 1.2):,}원 이하 완화 적용")
            else:
                filter_notes.append("예산 조건 후보가 적어 가격은 점수에만 반영")

    # 2) 거래데이터에서 카테고리 힌트 추출
    trade_hints, category_hints = get_trade_hints(query, top_k=trade_top_k)

    # 3) 상품 검색
    # 거래 힌트를 쿼리에 살짝 추가해 상품 검색을 보강
    augmented_query = query + " " + " ".join(category_hints)

    query_vec = product_vectorizer.transform([augmented_query])
    product_scores = cosine_similarity(query_vec, product_matrix).flatten()

    result_df = candidates.copy()

    # candidates의 index는 product_df 기준 index이므로 그대로 product_scores에서 조회
    result_df["base_score"] = result_df.index.map(lambda i: float(product_scores[i]))

    # 4) 거래 힌트 보너스
    def calc_trade_hint_bonus(row):
        text = combine_text(
            row.get("product_name", ""),
            row.get("category_path", ""),
            row.get("keywords", ""),
        )

        if any(hint and hint in text for hint in category_hints):
            return 0.08

        return 0.0

    result_df["trade_hint_bonus"] = result_df.apply(calc_trade_hint_bonus, axis=1)

    # 5) 예산 보너스
    if conditions["budget"] is not None:
        budget = conditions["budget"]

        def calc_budget_bonus(price):
            if pd.isna(price):
                return 0.0
            if price <= budget:
                return 0.05
            if price <= budget * 1.2:
                return 0.02
            return -0.03

        result_df["budget_bonus"] = result_df["price"].apply(calc_budget_bonus)
    else:
        result_df["budget_bonus"] = 0.0

    # 6) 수량 보너스
    if conditions["quantity"] is not None:
        qty = conditions["quantity"]

        result_df["qty_ok"] = result_df["moq"].apply(
            lambda x: True if pd.isna(x) else x <= qty
        )

        result_df["quantity_bonus"] = result_df["qty_ok"].apply(
            lambda ok: 0.03 if ok else -0.03
        )
    else:
        result_df["qty_ok"] = ""
        result_df["quantity_bonus"] = 0.0

    # 7) 최종 점수
    result_df["final_score"] = (
        result_df["base_score"] +
        result_df["trade_hint_bonus"] +
        result_df["budget_bonus"] +
        result_df["quantity_bonus"]
    )

    # 추천 이유 생성
    def make_reason(row):
        reasons = []

        if conditions["budget"] is not None and pd.notna(row["price"]):
            if row["price"] <= conditions["budget"]:
                reasons.append(f"예산 {conditions['budget']:,}원 이하 조건에 맞음")
            elif row["price"] <= conditions["budget"] * 1.2:
                reasons.append("예산보다 약간 높지만 후보 부족으로 검토 가능")
            else:
                reasons.append("예산 초과 가능성 있음")

        if conditions["quantity"] is not None and row["qty_ok"] != "":
            if row["qty_ok"]:
                reasons.append("요청 수량 기준 최소구매수량 충족 가능")
            else:
                reasons.append("최소구매수량 확인 필요")

        if row["trade_hint_bonus"] > 0:
            reasons.append("과거 거래데이터의 유사 상품군과 관련 있음")

        if row["base_score"] > 0.2:
            reasons.append("요청 문구와 상품명/카테고리/키워드 유사도 높음")
        elif row["base_score"] > 0.1:
            reasons.append("요청 문구와 일부 관련 있음")

        if not reasons:
            reasons.append("상품명/카테고리 기준 후보로 검토 가능")

        return " / ".join(reasons)

    result_df["recommend_reason"] = result_df.apply(make_reason, axis=1)

    output_cols = [
        "product_id",
        "product_name",
        "price",
        "moq",
        "category_path",
        "keywords",
        "base_score",
        "trade_hint_bonus",
        "budget_bonus",
        "quantity_bonus",
        "final_score",
        "qty_ok",
        "recommend_reason",
        "image",
    ]

    recommendations = (
        result_df
        .sort_values("final_score", ascending=False)
        .head(top_k)
        [output_cols]
        .reset_index(drop=True)
    )

    return {
        "query": query,
        "conditions": conditions,
        "filter_notes": filter_notes,
        "category_hints": category_hints,
        "trade_hints": trade_hints,
        "recommendations": recommendations,
    }


## 11. 추천 테스트

아래 질문을 바꿔가며 테스트합니다.


In [ ]:
# =========================
# 추천 테스트
# =========================

test_query = "병원 개원 답례품으로 3천원 이하 상품 추천해줘"

result = recommend_products(test_query, top_k=5)

print("[질문]")
print(result["query"])

print("\n[추출 조건]")
print(result["conditions"])

print("\n[필터 메모]")
for note in result["filter_notes"]:
    print("-", note)

print("\n[거래데이터 기반 카테고리 힌트]")
print(result["category_hints"])

print("\n[추천 상품]")
display(result["recommendations"])

print("\n[참고 거래 사례]")
display(result["trade_hints"][[
    "trade_score",
    "buyer_type",
    "buyer_name",
    "trade_product_name",
    "trade_category_path",
    "print_method",
]].head(5))


## 12. 여러 질문 일괄 테스트

추천 결과를 엑셀로 저장합니다.


In [ ]:
# =========================
# 여러 질문 테스트
# =========================

test_queries = [
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    "대학교 행사에서 나눠줄 2천원대 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 저렴한 홍보물 추천해줘",
    "여름 행사에 어울리는 판촉물 추천해줘",
    "겨울철 고객 사은품 추천해줘",
    "500개 정도 주문할 수 있는 물티슈 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
    "예산 5천원 이하로 고급스러운 기념품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
]

all_rows = []

for query in test_queries:
    result = recommend_products(query, top_k=5)

    for rank, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "base_score": row["base_score"],
            "trade_hint_bonus": row["trade_hint_bonus"],
            "budget_bonus": row["budget_bonus"],
            "quantity_bonus": row["quantity_bonus"],
            "final_score": row["final_score"],
            "qty_ok": row["qty_ok"],
            "recommend_reason": row["recommend_reason"],
            "category_hints": ", ".join(result["category_hints"]),
            "filter_notes": " / ".join(result["filter_notes"]),
        })

batch_result_df = pd.DataFrame(all_rows)

BATCH_RESULT_PATH = OUTPUT_DIR / "reco_v0_tfidf_batchtest.xlsx"
batch_result_df.to_excel(BATCH_RESULT_PATH, index=False)

print("추천 테스트 결과 저장:", BATCH_RESULT_PATH)
display(batch_result_df.head(20))


## 13. Ollama로 추천 설명 생성하기

선택 사항입니다.

사전 준비:

```bash
pip install ollama
ollama pull gemma3
```

또는 한국어가 더 나은 모델을 쓰고 싶으면:

```bash
ollama pull qwen3
```

이 셀은 추천 상품 리스트를 바탕으로 고객에게 보여줄 자연어 답변을 생성합니다.


In [ ]:
# =========================
# Ollama 답변 생성
# =========================

def build_recommendation_context(result: dict) -> str:
    recs = result["recommendations"]
    hints = result["trade_hints"].head(5)

    lines = []

    lines.append("[추출 조건]")
    lines.append(str(result["conditions"]))
    lines.append("")

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(recs.iterrows(), start=1):
        lines.append(f"{i}. {row['product_name']}")
        lines.append(f"   - 가격: {row['price']}")
        lines.append(f"   - 최소구매수량: {row['moq']}")
        lines.append(f"   - 카테고리: {row['category_path']}")
        lines.append(f"   - 추천근거: {row['recommend_reason']}")
        lines.append("")

    lines.append("[참고 거래 사례]")
    for i, (_, row) in enumerate(hints.iterrows(), start=1):
        lines.append(f"{i}. {row['trade_product_name']}")
        lines.append(f"   - 구매처유형: {row['buyer_type']}")
        lines.append(f"   - 상품분류: {row['trade_category_path']}")
        lines.append("")

    return "\n".join(lines)


def generate_recommendation_answer_with_ollama(
    query: str,
    model: str = "gemma3",
    top_k: int = 5,
):
    try:
        import ollama
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "ollama 패키지가 설치되어 있지 않습니다. "
            "현재 가상환경에서 `python -m pip install ollama`를 실행하세요."
        )

    result = recommend_products(query, top_k=top_k)
    context = build_recommendation_context(result)

    system_prompt = """
너는 쇼핑몰 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보와 거래 사례만 근거로 답변한다.
- 가격, 최소구매수량, 납기 등 데이터에 없는 내용은 단정하지 않는다.
- 고객에게는 Top 3~5개 상품을 간단히 추천한다.
- 각 상품마다 추천 이유를 붙인다.
- 조건이 부족하면 추가 확인이 필요한 항목을 마지막에 적는다.
- 자연스러운 한국어로 답변한다.
"""

    user_prompt = f"""
[사용자 요청]
{query}

[추천 근거 데이터]
{context}

위 근거만 바탕으로 고객에게 상품 추천 답변을 작성해줘.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()},
        ],
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
        },
        stream=False,
    )

    return {
        "query": query,
        "answer": response["message"]["content"],
        "result": result,
    }

In [ ]:
ollama_result = generate_recommendation_answer_with_ollama(
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    model="gemma3:4b",
    top_k=5,
)
print(ollama_result["answer"])
display(ollama_result["result"]["recommendations"])

# 다음 단계

이 v0를 실행한 뒤 확인할 것:

1. 가격 필터가 과하게 작동하지 않는가?
2. 거래데이터 힌트가 상품 추천을 오히려 이상하게 만들지는 않는가?
3. 상품명이 아닌 카테고리 기준으로도 추천이 잘 되는가?
4. 상품데이터의 `검색키워드`, `카테고리`, `상품내용` 품질이 충분한가?
5. 추천 결과에 실제로 판매 가능한 상품이 나오는가?

## 다음 버전

다음: v1에서 질문을 구조화하고, 거래데이터를 임베딩 기반으로 검색합니다. 전체 버전 로드맵과 배경은 `README.md`를 참고하세요.
